In [1]:
import asyncio
import aiohttp
import pandas as pd
import datetime
import math
import os

## 1. Obtenemos los datos de la api

In [2]:
async def fetch(session, params):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "es-MX,es;q=0.9",
        "Referer": "https://www.datos.gob.mx/",
    }
    url = "https://www.datos.gob.mx/api/3/action/current_package_list_with_resources"
    async with session.get(url, headers=headers, params=params) as response:
        if response.status == 200:
            result = await response.json()
            return result['result']
        else:
            return response.error

async def multiFetch():
    maxExpectedPackages = 2500
    block_size = 100
    numRequests = math.ceil(maxExpectedPackages / block_size) 
    params = list()
    conjuntos = []
    for i in range(numRequests):
        if i == 0:
            param = {"limit" : block_size - 1 , "offset" : i * block_size}
        else:
            param = {"limit" : block_size -1 , "offset" : i * block_size}
        params.append(param)

    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, param) for param in params]
        resultados = await asyncio.gather(*tasks, return_exceptions=True)

    for resultado in resultados:
        conjuntos += resultado

    return conjuntos

In [3]:
def iterar_recursos(conjuntos):
    recursos = []
    for conjunto in conjuntos:
        for recurso in conjunto["resources"]:
            datum = {
                "nombre_categoria": conjunto["groups"][0]["display_name"],
                "clave_categoria": conjunto["groups"][0]["name"],
                "descripcion_categoria": conjunto["groups"][0]["description"],
                "id_categoria": conjunto["groups"][0]["id"],
                "nombre_institucion": conjunto["organization"]["title"],
                "descripcion_institucion": conjunto["organization"]["description"],
                "nombre_paquete": conjunto["title"],
                "clave_paquete": conjunto["name"],
                "notas_paquete": conjunto["notes"],
                "etiquetas": list(map(lambda x: x["name"], conjunto["tags"])),
                "id_paquete": conjunto["id"],
                "id_recurso": recurso["id"],
                "nombre_recurso": recurso["name"],
                "descripcion_recurso": recurso["description"],
                "url_recurso": recurso["url"],
                "frecuencia_actualizacion": recurso["update_frequency"],
                "creacion_recurso": recurso["created"],
                "ultima_modificacion": recurso["last_modified"],
                "modificacion_metadatos": recurso["metadata_modified"]
            }
            recursos.append(datum)
    return recursos

In [4]:
data_ckan = await multiFetch()
len(data_ckan)

1735

In [5]:
conjuntos_df = pd.DataFrame(data_ckan)
conjuntos_df.columns

Index(['author', 'author_email', 'creator_user_id', 'id', 'isopen',
       'license_id', 'license_title', 'license_url', 'maintainer',
       'maintainer_email', 'metadata_created', 'metadata_modified', 'name',
       'notes', 'num_resources', 'num_tags', 'organization', 'owner_org',
       'private', 'state', 'title', 'type', 'url', 'version', 'groups',
       'resources', 'tags', 'extras', 'relationships_as_subject',
       'relationships_as_object'],
      dtype='str')

## 2. Iteramos sobre los recursos para obtener su información

In [6]:
recursos_ckan = iterar_recursos(data_ckan)
len(recursos_ckan)

7773

In [7]:
recursos_df = pd.DataFrame(recursos_ckan)
recursos_df.shape

(7773, 19)

In [8]:
recursos_df.columns

Index(['nombre_categoria', 'clave_categoria', 'descripcion_categoria',
       'id_categoria', 'nombre_institucion', 'descripcion_institucion',
       'nombre_paquete', 'clave_paquete', 'notas_paquete', 'etiquetas',
       'id_paquete', 'id_recurso', 'nombre_recurso', 'descripcion_recurso',
       'url_recurso', 'frecuencia_actualizacion', 'creacion_recurso',
       'ultima_modificacion', 'modificacion_metadatos'],
      dtype='str')

In [9]:
recursos_df.dtypes

nombre_categoria               str
clave_categoria                str
descripcion_categoria          str
id_categoria                   str
nombre_institucion             str
descripcion_institucion        str
nombre_paquete                 str
clave_paquete                  str
notas_paquete                  str
etiquetas                   object
id_paquete                     str
id_recurso                     str
nombre_recurso                 str
descripcion_recurso            str
url_recurso                    str
frecuencia_actualizacion       str
creacion_recurso               str
ultima_modificacion            str
modificacion_metadatos         str
dtype: object

In [10]:
recursos_df.loc[0]

nombre_categoria                                                      Energía
clave_categoria                                                       energia
descripcion_categoria       Datos sobre la producción y distribución de en...
id_categoria                             ca050a05-ac4b-43ce-aab9-ff9f20520fba
nombre_institucion                              Secretaría de Energía (SENER)
descripcion_institucion     Órgano de la Administración Pública Federal Ce...
nombre_paquete              Contratos para la Exploración y Extracción de ...
clave_paquete               contratos_para_exploracion_extraccion_hidrocar...
notas_paquete               Información sobre la comercialización de los h...
etiquetas                   [actividad petrolera, contrato, explotación, e...
id_paquete                               f91adbca-796a-44a6-8038-5c617092b86e
id_recurso                               1b4df48f-9da6-409b-8390-8b6fdfa95a9f
nombre_recurso              Contraprestaciones de CEE a favor de

In [11]:
recursos_df["creacion_recurso"] = pd.to_datetime(recursos_df["creacion_recurso"])
recursos_df["ultima_modificacion"] = pd.to_datetime(recursos_df["ultima_modificacion"])
recursos_df["modificacion_metadatos"] = pd.to_datetime(recursos_df["modificacion_metadatos"])

In [12]:
recursos_df.dtypes

nombre_categoria                       str
clave_categoria                        str
descripcion_categoria                  str
id_categoria                           str
nombre_institucion                     str
descripcion_institucion                str
nombre_paquete                         str
clave_paquete                          str
notas_paquete                          str
etiquetas                           object
id_paquete                             str
id_recurso                             str
nombre_recurso                         str
descripcion_recurso                    str
url_recurso                            str
frecuencia_actualizacion               str
creacion_recurso            datetime64[us]
ultima_modificacion         datetime64[us]
modificacion_metadatos      datetime64[us]
dtype: object

In [13]:
recursos_df[recursos_df["ultima_modificacion"] == recursos_df["modificacion_metadatos"]]

,nombre_categoria,clave_categoria,descripcion_categoria,id_categoria,nombre_institucion,descripcion_institucion,nombre_paquete,clave_paquete,notas_paquete,etiquetas,id_paquete,id_recurso,nombre_recurso,descripcion_recurso,url_recurso,frecuencia_actualizacion,creacion_recurso,ultima_modificacion,modificacion_metadatos


In [14]:
recursos_df["ultima_modificacion"].isna().value_counts()

ultima_modificacion
True     7348
False     425
Name: count, dtype: int64

In [15]:
recursos_df["modificacion_metadatos"].isna().value_counts()

modificacion_metadatos
False    7773
Name: count, dtype: int64

In [16]:
recursos_df[recursos_df["modificacion_metadatos"] > recursos_df["creacion_recurso"]].shape

(7768, 19)

## 3. Les agregamos una columna con los años de creación

In [17]:
recursos_df["anio_creacion"] = recursos_df["creacion_recurso"].dt.year
recursos_df["anio_ultima_modificacion"] = recursos_df["ultima_modificacion"].dt.year

## 4. Les agregamos la semana de creación y la semana de modificación

In [18]:
recursos_df["semana_creacion"] = recursos_df["creacion_recurso"].dt.isocalendar().week
recursos_df["semana_ultima_actualizacion"] = recursos_df["ultima_modificacion"].dt.isocalendar().week

## 6. Obtenemos el agregado semanal de recursos nuevos subidos

In [19]:
subidos_semanales = recursos_df[["anio_creacion", "semana_creacion"]].groupby(["anio_creacion", "semana_creacion"]).size().reset_index()
subidos_semanales = subidos_semanales.rename(columns = {0: 'recursos_agregados'})
subidos_semanales = subidos_semanales.sort_values(by = ["anio_creacion", "semana_creacion"], ascending = False)
subidos_semanales

,anio_creacion,semana_creacion,recursos_agregados
70,2026,40,4
69,2026,39,77
68,2026,38,128
67,2026,37,69
66,2026,36,101
...,...,...,...
4,2025,15,3
3,2025,12,8
2,2025,11,475
1,2025,10,1


In [20]:
subidos_semanales["recursos_agregados"].sum()

np.int64(7773)

## 7. Eliminamos el reporte del día anterior

In [21]:
hoy = datetime.datetime.today()
hoy = pd.to_datetime(hoy)
hoy

Timestamp('2026-09-29 09:30:48.960921')

In [22]:
ayer = hoy - datetime.timedelta(days = 1)
ayer

Timestamp('2026-09-28 09:30:48.960921')

In [23]:
nombre_ayer = f"ckan_{ayer.day}_{ayer.month_name()}_{ayer.year}"
nombre_ayer

'ckan_28_September_2026'

In [25]:
try:
    os.remove(f"{nombre_ayer.csv}")
except:
    print("No hay archivo para borrar")

No hay archivo para borrar


## 8. Generamos el csv nuevo

In [26]:
nombre_hoy = f"ckan_{hoy.day}_{hoy.month_name()}_{hoy.year}"
nombre_hoy

'ckan_29_September_2026'

In [ ]:
subidos_semanales.to_csv(f"{nombre_hoy}.csv")